#### This workbook contains the code needed to combine the Alpha Vantage API queries and Google RSS feed headline sentiment functions into one dataset, which is stored in an SQL database

First, we install the dependencies needed for the code

In [2]:
#dependencies for the Google RSS Headline Sentiment Analysis
!pip install --quiet torch transformers
import requests
import numpy as np
from bs4 import BeautifulSoup
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import warnings
warnings.filterwarnings('ignore')

Following load_model and headline_sentiment functions are required for headline sentiment analysis

In [3]:
#function to load NLP Model from Huggingface
def load_model(source_model):
    source_model = source_model
    tokenizer = AutoTokenizer.from_pretrained(source_model)
    model = AutoModelForSequenceClassification.from_pretrained(source_model)
    return tokenizer, model

In [4]:
#function for Google RSS Feed Headline Sentiment Analysis
def headline_sentiment(query, date, tokenizer, model):
    #create url
    base_url = 'https://news.google.com/rss/search?q=us'
    string_query = query
    split_string = string_query.split(' ')
    for word in split_string:
        base_url += f'%20{word}'
    date_str = (date).split('-')
    month = date_str[1]
    day = date_str[2]
    year = date_str[0]
    month_url = f'%20when%3A{month}'
    day_url = f'%2F{day}'
    year_url = f'%2F{year}'
    country_and_lang = '&hl=en-US&gl=US&ceid=US%3Aen'
    final_url = base_url + month_url + day_url + year_url + country_and_lang

    #conduct search with url and return dataframe with Date and Headline. Must filter for exact date.
    url = final_url
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'xml')
    headlines = soup.find('channel').find_all('title')
    hlist = []
    dlist = []
    headlines = soup.find('channel').find_all('title')
    dates = soup.find('channel').find_all('pubDate')
    for item in headlines:
        hlist.append(item.get_text())
    for item in dates:
        dlist.append(item.get_text())
    del hlist[0:2]
    date_hl_df = pd.DataFrame({
    'Dates':dlist, 'Headlines':hlist})
    date_hl_df['Dates'] = pd.to_datetime(date_hl_df['Dates'])
    date_hl_df['Dates'] = date_hl_df['Dates'].dt.strftime('%Y-%m-%d')
    date_hl_df = date_hl_df[date_hl_df['Dates'] == date]
    
    #define sentiment function
    def get_sentiment(example):
        encoded_text = tokenizer(example, return_tensors = 'pt')
        output = model(**encoded_text)
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        scores_dict = {
            'finbert_pos' : scores[0],
            'finbert_neg' : scores[1],
            'finbert_neu' : scores[2]
        }
        return scores_dict
    
    #apply sentiment function and return mean sentiment of each type
    hl_sentiment = date_hl_df['Headlines'].apply(get_sentiment)
    finbert_neg = []
    for row in hl_sentiment:
        finbert_neg.append(list(row.values())[0])
    finbert_neut = []
    for row in hl_sentiment:
        finbert_neut.append(list(row.values())[1])
    finbert_pos = []
    for row in hl_sentiment:
        finbert_pos.append(list(row.values())[2])
    final_hl_df = pd.DataFrame({
    'Headline': date_hl_df['Headlines'],
    'Positive finBERT Score': finbert_neg,
    'Negative finBERT Score': finbert_neut,
    'Neutral finBERT Score': finbert_pos
    })
    avg_pos = final_hl_df['Positive finBERT Score'].mean()
    avg_neg = final_hl_df['Negative finBERT Score'].mean()
    avg_neut = final_hl_df['Neutral finBERT Score'].mean()
    summary_table = pd.DataFrame({
    'Date':date,
    'Mean Pos Sentiment':avg_pos,
    'Mean Neg Sentiment':avg_neg,
    'Mean Neut Sentiment':avg_neut},
                                index = [0])
    return summary_table

In [5]:
#get tokenizer and model from Huggingface model
tokenizer, model = load_model('ProsusAI/finbert')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
#define search term and search date for the headline search.
#Specify tokenizer and headline to use. Outputs average sentiments scores for that topic and day.
headline_sentiment('Economy', '2026-04-18', tokenizer, model)

,Date,Mean Pos Sentiment,Mean Neg Sentiment,Mean Neut Sentiment
0,2026-04-18,0.154424,0.229015,0.616562


Following function makes API request to Alpha Vantage API to get historical stock data for specific stock (i.e., stock symbol). NOTE: Future users will need thier own API key.

In [7]:
def get_stock_info(function, symbol, api_key):
    base_url = 'https://www.alphavantage.co/query?'
    function = f'function={function}&'
    symbol = f'symbol={symbol}&'
    api_key = f'apikey={api_key}'
    query_url = base_url + function + symbol + api_key
    response = requests.get(query_url)
    data = response.json()
    stock_df = pd.DataFrame(data['Time Series (Daily)']).transpose()
    stock_df = stock_df.rename(columns = {'1. open':'Open', '2. high':'High', '3. low':'Low', '4. close':'Close', '5. volume':'Volume'})
    reset_index = stock_df.reset_index()
    reset_index.rename(columns = {'index':'Date'}, inplace = True)
    reset_index = reset_index.astype({'Open':'float', 'High':'float','Low':'float','Close':'float','Volume':'int'})
    reset_index['Date'] = pd.to_datetime(reset_index['Date'], format = 'ISO8601')
    final_df = reset_index
    return final_df

In [34]:
#make a query for Apple (AAPL) for daily time series data with your API key. Using iloc to make a smaller data table for testing
AAPL_df = get_stock_info('TIME_SERIES_DAILY', 'AAPL', 'XXXXX')
AAPL_df = AAPL_df.sort_values(by = 'Date')
AAPL_df_short = AAPL_df.iloc[0:10,:]
AAPL_df_short

,Date,Open,High,Low,Close,Volume
99,2025-11-21,265.950,273.33,265.6700,271.49,59030832
98,2025-11-24,270.900,277.00,270.9000,275.92,65585796
97,2025-11-25,275.270,280.38,275.2500,276.97,46914220
96,2025-11-26,276.960,279.53,276.6300,277.55,33431423
95,2025-11-28,277.260,279.00,275.9865,278.85,20135620
94,2025-12-01,278.010,283.42,276.1400,283.10,46587722
93,2025-12-02,283.000,287.40,282.6301,286.19,53669532
92,2025-12-03,286.200,288.62,283.3000,284.15,43538687
91,2025-12-04,284.095,284.73,278.5900,280.70,43989056
90,2025-12-05,280.540,281.14,278.0500,278.78,47265845


We will capture the date range from the API output and iterate through the range to get the headline sentiment from every date in the range

In [35]:
#testing iteration through query dates list
sentiment_table = pd.DataFrame({
    'Date':'',
    'Mean Pos Sentiment':'',
    'Mean Neg Sentiment':'',
    'Mean Neut Sentiment':''
}, index = [0])
query_dates = AAPL_df_short['Date']
query_dates = query_dates.astype(str)
for date in query_dates:
    sentiment_table = pd.concat([sentiment_table, headline_sentiment('Economy', date , tokenizer, model)], ignore_index = True) 
sentiment_table.drop(0)

,Date,Mean Pos Sentiment,Mean Neg Sentiment,Mean Neut Sentiment
1,2025-11-21,0.201534,0.242864,0.555602
2,2025-11-24,0.171835,0.149069,0.679095
3,2025-11-25,0.217654,0.278674,0.503672
4,2025-11-26,0.152934,0.322406,0.52466
5,2025-11-28,0.132896,0.18719,0.679914
6,2025-12-01,0.137593,0.198872,0.663535
7,2025-12-02,0.182325,0.264339,0.553336
8,2025-12-03,0.160912,0.071917,0.76717
9,2025-12-04,0.14253,0.254134,0.603336
10,2025-12-05,0.142929,0.269399,0.587673


In [43]:
#make a query for Apple (AAPL) for full daily time series data with your API key (will return 100 rows with the non-premium version of Alpha Vantage)
AAPL_df = get_stock_info('TIME_SERIES_DAILY', 'AAPL', 'XXXXXX')
AAPL_df = AAPL_df.sort_values(by = 'Date')
AAPL_df['Date'] = AAPL_df['Date'].astype('str')
AAPL_df

,Date,Open,High,Low,Close,Volume
99,2025-11-21,265.950,273.33,265.6700,271.49,59030832
98,2025-11-24,270.900,277.00,270.9000,275.92,65585796
97,2025-11-25,275.270,280.38,275.2500,276.97,46914220
96,2025-11-26,276.960,279.53,276.6300,277.55,33431423
95,2025-11-28,277.260,279.00,275.9865,278.85,20135620
...,...,...,...,...,...,...
4,2026-04-13,259.730,260.18,256.6600,259.20,36234698
3,2026-04-14,259.245,261.93,257.1900,258.83,48370710
2,2026-04-15,258.160,266.56,257.8100,266.43,49913510
1,2026-04-16,266.800,267.16,261.2700,263.40,43323112


In [36]:
#iterate through full 100 dates obtained from API
sentiment_table = pd.DataFrame({
    'Date':'',
    'Mean Pos Sentiment':'',
    'Mean Neg Sentiment':'',
    'Mean Neut Sentiment':''
}, index = [0])
query_dates = AAPL_df['Date']
query_dates = query_dates.astype(str)
for date in query_dates:
    sentiment_table = pd.concat([sentiment_table, headline_sentiment('Economy', date , tokenizer, model)], ignore_index = True) 
sentiment_table = sentiment_table.drop(0)

In [40]:
sentiment_table

,Date,Mean Pos Sentiment,Mean Neg Sentiment,Mean Neut Sentiment
1,2025-11-21,0.201534,0.242864,0.555602
2,2025-11-24,0.171835,0.149069,0.679095
3,2025-11-25,0.217654,0.278674,0.503672
4,2025-11-26,0.152934,0.322406,0.52466
5,2025-11-28,0.132896,0.18719,0.679914
...,...,...,...,...
96,2026-04-13,0.046506,0.148978,0.804516
97,2026-04-14,0.178496,0.156727,0.664777
98,2026-04-15,0.055153,0.091633,0.853213
99,2026-04-16,0.173285,0.140736,0.685979


In [47]:
#join AAPL_df API table and sentiment table on date column
AAPL_merged = AAPL_df.merge(sentiment_table, on = 'Date', how = 'inner')
AAPL_merged

,Date,Open,High,Low,Close,Volume,Mean Pos Sentiment,Mean Neg Sentiment,Mean Neut Sentiment
0,2025-11-21,265.950,273.33,265.6700,271.49,59030832,0.201534,0.242864,0.555602
1,2025-11-24,270.900,277.00,270.9000,275.92,65585796,0.171835,0.149069,0.679095
2,2025-11-25,275.270,280.38,275.2500,276.97,46914220,0.217654,0.278674,0.503672
3,2025-11-26,276.960,279.53,276.6300,277.55,33431423,0.152934,0.322406,0.52466
4,2025-11-28,277.260,279.00,275.9865,278.85,20135620,0.132896,0.18719,0.679914
...,...,...,...,...,...,...,...,...,...
95,2026-04-13,259.730,260.18,256.6600,259.20,36234698,0.046506,0.148978,0.804516
96,2026-04-14,259.245,261.93,257.1900,258.83,48370710,0.178496,0.156727,0.664777
97,2026-04-15,258.160,266.56,257.8100,266.43,49913510,0.055153,0.091633,0.853213
98,2026-04-16,266.800,267.16,261.2700,263.40,43323112,0.173285,0.140736,0.685979


#### We now have our final AAPL_merged data frame. We will now use sqlalchemy to connect to our MySQL database and store the info for later use.

In [49]:
#installing dependencies
!pip install --quiet sqlalchemy mysql-connector-python
import sqlalchemy

In [53]:
#creating engine for sql connection. Anonymonized values used for SQL connection
user = 'Username'
password = 'Password'
host = 'YourPC'
database = 'stocks'

engine = sqlalchemy.create_engine(f'mysql+mysqlconnector://{user}:{password}@{host}/{database}')
try:
    with engine.connect() as connection:
        print('Connected!')
except:
    print('Something went wrong')

Connected!


In [54]:
AAPL_merged.to_sql(name = 'aapl', con = engine, if_exists = 'replace', index = False)

100

#### This results in the data being stored in an SQL database for future use in modeling.

Two important future steps:
1) Adapting function to only add data for dates that are missing from the currently existing SQL table. For example, if the date is 4/18/26 and the last data point is for 4/15/2026, the algorithm should only append the missing dates 4/16, 4/17, and 4/18 to the SQL database.
2) Using compiled data for prediction.